# 03. CIFAR-100, five deletions, four methods (M3)

This notebook runs the paper's experiment on the paper's own dataset for the first time.

It answers one question that the project has been carrying since the start. When the method
trains a small patch to delete a class, how much of the model's limited input space does that
patch use up? The paper's own figure implies about 2.5 directions out of 64. The paper's own
equations imply exactly 4. Those two numbers cannot both be right, and five deletions are enough
to tell them apart.

Along the way it also compares the four methods, which is the milestone's stated job.

Runs on CPU in a few minutes. No GPU needed.

## Words used in this notebook

Read this once and the rest will follow. Nothing here assumes you have read the paper.

- **Backbone.** A trained image network, frozen. We never change it. We only use the 64 numbers
  it produces for each image, which are its summary of what the image contains.
- **Head.** The last layer, which turns those 64 numbers into 100 scores, one per CIFAR-100
  class. A 64 by 100 matrix. This is the only thing unlearning touches.
- **Adapter.** A small correction to the head, built as two thin matrices multiplied together:
  `A` is 64 by 4, `B` is 4 by 100. Their product is 64 by 100, the same shape as the head, but it
  is built from only 4 independent directions. One adapter is trained per deletion.
- **Rank.** How many independent directions a matrix actually uses. The adapter is built to use
  4. Whether a *trained* adapter really uses all 4 is the question this notebook answers.
- **Deletion task.** One request to make the model forget one class. We do five, one after
  another, and the model never sees the earlier classes' data again.
- **Input subspace.** The 64 directions the head reads from. Each adapter claims 4 of them, and
  the method forbids every later adapter from reading the claimed ones.
- **Projection.** The matrix `P` that blocks the claimed directions. A new adapter must build
  itself from `P` times something, so it can only read what is still free.
- **Free rank.** How many of the 64 directions are still unclaimed. It starts at 64 and falls as
  tasks arrive. When it hits 0 a new adapter has nothing left to read and its correction is
  exactly zero.
- **Budget.** 64 divided by 4, so 16 tasks, if every adapter really uses 4. Five deletions spends
  5 of the 16, so this notebook cannot show the budget running out. It can only show the rate.
- **Fusion.** How the deletions are applied: subtract all the adapters from the head at once.
- **Lambda.** How hard to subtract. 0 means do nothing. Larger means forget harder and risk
  damaging the classes you are keeping.
- **Retain accuracy.** Accuracy on the classes we are keeping. Higher is better.
- **Forget accuracy.** Accuracy on the classes we deleted. Lower is better. We report it two
  ways, because the paper never says which one it means: on the class deleted most recently, and
  averaged over every class deleted so far.
- **Router.** A fourth method that keeps all the adapters and lets a small gate pick between them
  per image, instead of subtracting anything.

## 0. Setup on Colab

Clones this project, installs it, and moves into it. On a local machine it does nothing and uses
the checkout you already have.

The guard at the top matters. If the project is already imported, Colab will keep running the old
copy even after the clone succeeds, and you would silently test stale code. Restart the runtime
if it stops you.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/sumitasthana/DSR.git"
REVISION = "main"   # Replace with a commit hash to pin this run.

if "src" in sys.modules:
    raise RuntimeError("The experiment package is already imported. "
                       "Restart the runtime before running setup, or you will "
                       "silently test stale code.")

if "google.colab" in sys.modules:
    if not Path("DSR").exists():
        subprocess.run(["git", "clone", REPO_URL], check=True)
    subprocess.run(["git", "-C", "DSR", "fetch", "--all", "--quiet"], check=True)
    subprocess.run(["git", "-C", "DSR", "checkout", REVISION], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "DSR"], check=True)
    os.chdir("DSR")
    print("cloned and installed")
else:
    print("not on Colab; using the existing checkout")

print("working directory:", Path.cwd())

## 1. Load code

Imports the project and prints which exact version is about to run. The commit hash and the
"local changes" line are what make a result citable later: a number from a dirty tree cannot be
reproduced by anyone else.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import src.common as common
from src.common import load_config, provenance, save_json
from src.run_m3 import run_m3
from src.unlearn import METHODS

REPO = Path(common.__file__).resolve().parents[1]
PROV = provenance()

print("repo          :", REPO)
print("commit        :", PROV["commit"])
print("local changes :", PROV["git_status"] or "none")
print("torch         :", PROV["torch"], "| python", PROV["python"].split()[0])

## 2. Settings

Everything the run depends on comes from `config.yaml`, so there are no numbers hidden in this
notebook. The printout below is the whole experiment stated in advance: which classes get
deleted, how hard, how many times over, and how much work that adds up to.

The capacity line is the one to watch. It is the budget the next milestone will actually spend.

In [ ]:
CFG = load_config(REPO / "config.yaml")
CLASSES = CFG["cifar100"]["forget_classes"]
LAMBDAS = CFG["unlearn"]["lambdas"]
SEEDS = CFG["seeds"]
RANK = CFG["projection"]["rank"]
D_IN = CFG["model"]["feature_dim"]
RUN = "nb03-cifar100-n5"

print("resolved unlearn block :", json.dumps(CFG["unlearn"]))
print()
print(f"forget order  : {CLASSES}  (N = {len(CLASSES)})")
print(f"methods       : {list(METHODS)}")
print(f"lambdas       : {LAMBDAS}")
print(f"seeds         : {SEEDS}")
print(f"head          : {D_IN} x {CFG['model']['classes']}, rank {RANK}")
print(f"capacity      : {D_IN} / {RANK} = {D_IN // RANK} tasks, so N = {len(CLASSES)} spends "
      f"{len(CLASSES)} of {D_IN // RANK}")
print(f"planned work  : {len(SEEDS) * len(METHODS) * len(CLASSES)} adapters, {len(SEEDS)} routers")
print(f"evaluations   : {len(SEEDS) * len(METHODS) * len(CLASSES) * len(LAMBDAS)} rows")
print("precision     : float64 throughout (see ASSUMPTIONS.md M3)")
print("hardware      : CPU only. A few minutes, plus feature extraction on a first run.")

## 3. Connect to storage

Colab throws away everything when the session ends, so results are copied to object storage. If
the credentials are missing the notebook still runs end to end, it just keeps no copy, and
section 8 says so rather than failing quietly.

In [ ]:
from src.runstore import connect

OUT = REPO / "outputs" / RUN
OUT.mkdir(parents=True, exist_ok=True)
print("writing results to", OUT)

STORE = connect()
if STORE is None:
    print()
    print("No R2 credentials found. This run will produce results but keep no copy.")
    print("Set DSR_ENDPOINT, DSR_ACCESS_KEY_ID and DSR_SECRET_ACCESS_KEY to keep it.")
else:
    print("R2 bucket    :", STORE.bucket)
    try:
        print("runs on file :", ", ".join(STORE.runs()) or "none yet")
    except Exception as error:
        print("runs on file : could not list,", type(error).__name__)

## 4. Run

One call does the whole experiment, because the loop lives in the package where it is tested
rather than in this notebook where it is not.

What it does, in order: downloads CIFAR-100 and the frozen backbone if they are not already
here, pushes all 60,000 images through the backbone once and caches the 64 numbers per image,
then for each seed trains a router and runs all four methods through the five deletions in turn.
Every trained sequence is then scored at every lambda, which costs nothing extra because lambda
only enters at fusion time and never during training.

The first run is slower than later ones because of the feature extraction. After that the cache
is verified by checksum and reused.

In [ ]:
REPORT = run_m3(CFG)

metrics = pd.DataFrame(REPORT["metrics"])
subspace = pd.DataFrame(REPORT["subspace"])
norms = pd.DataFrame(REPORT["fusion_norms"])
print()
print(f"{len(metrics)} metric rows, {len(subspace)} subspace rows, {len(norms)} norm rows")

## 5. The measurement this notebook exists for

This is test T1 in the project's spec, and the rule for reading it was fixed on 8 November, before
any of this code ran:

> If the reserved rank is 4, the budget is 16 tasks and the paper's Figure 3 is wrong. If it is
> near 2.5, Figure 3 is right and the projection is reserving directions the adapter never used.

`reserved_rank` is how many directions that task's trained adapter actually claimed.
`free_rank_after` is how many of the 64 were still unclaimed once it had.

In [ ]:
used = (subspace.groupby(["method", "task", "forget_class"])
        [["reserved_rank", "free_rank_after"]].agg(["mean", "std"]).round(3))
print(used.to_string())

distinct = sorted(subspace["reserved_rank"].unique())
budget = D_IN // RANK
print()
print(f"distinct reserved ranks across every task, method and seed: {distinct}")

FULL_RANK = distinct == [RANK]
if FULL_RANK:
    print(f"Every task consumed the full rank {RANK}.")
    print(f"Capacity is {D_IN} / {RANK} = {budget} tasks, measured rather than assumed.")
    print(f"The paper's ~23 free directions near task 16 would need {(64 - 23) / 16:.2f} per task.")
    print("That is not reachable from these updates.")
else:
    print(f"Not always {RANK}. The paper's Figure 3 becomes reachable and the spec needs rewriting.")

### How fast the space runs out

Free rank against task number, with the budget drawn in. Five tasks is a short line, and that is
the point: it shows the slope, and the slope is what decides whether 20 or 30 deletions were ever
possible.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

line = subspace[subspace["seed"] == SEEDS[0]].groupby("task")["free_rank_after"].mean()
axes[0].plot([0, *line.index], [D_IN, *line.values], marker="o")
axes[0].plot([0, budget], [D_IN, 0], linestyle="--", color="grey",
             label=f"exhausted at task {budget}")
axes[0].set_xlabel("deletions so far")
axes[0].set_ylabel("directions still free")
axes[0].set_title(f"Free space, rank {RANK} per task")
axes[0].set_ylim(0, D_IN + 2)
axes[0].legend()

for strength in [l for l in LAMBDAS if l > 0]:
    part = norms[(norms["seed"] == SEEDS[0]) & (norms["lambda"] == strength)]
    axes[1].plot(part["task"], part["fused_delta_norm"], marker="o", label=f"lambda {strength}")
axes[1].set_xlabel("deletions so far")
axes[1].set_ylabel("size of the total correction")
axes[1].set_title("How much the head is being changed")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 6. The four methods, after all five deletions

Retain accuracy is on the 95 classes we kept, so higher is better. Forget accuracy is on the 5 we
deleted, so lower is better. Both forget readings are shown because the paper never says which
one it uses, and they disagree.

Lambda 0 is the untouched model, which is the only honest baseline to read the rest against.

In [ ]:
final = metrics[metrics["tasks"] == len(CLASSES)]
table = (final.groupby(["method", "lambda"])
         [["retain_excluding_forgotten", "forget_current", "forget_all"]]
         .agg(["mean", "std"]).round(2))
print(table.to_string())

BASELINE = final[(final["method"] == "static") & (final["lambda"] == 0.0)]
print()
print(f"untouched model: retain {BASELINE['retain_excluding_forgotten'].mean():.2f}%, "
      f"forget {BASELINE['forget_all'].mean():.2f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for method in METHODS:
    part = final[final["method"] == method].groupby("lambda")
    axes[0].plot(part.groups.keys(),
                 part["retain_excluding_forgotten"].mean(), marker="o", label=method)
    axes[1].plot(part.groups.keys(), part["forget_all"].mean(), marker="o", label=method)

axes[0].set_title("Accuracy kept on the 95 classes we are keeping")
axes[1].set_title("Accuracy left on the 5 classes we deleted")
for ax in axes:
    ax.set_xlabel("lambda")
    ax.set_ylabel("accuracy %")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 7. Check against the rules fixed before the run

The milestone's rule, from the project's instructions section 8, has four clauses. Each is
checked here as a plain boolean so the result cannot be talked into agreeing afterwards.

In [ ]:
def at(method, strength, column):
    rows = final[(final["method"] == method) & (final["lambda"] == strength)]
    return float(rows[column].mean())


top = max(LAMBDAS)
STATIC_DEGRADES = at("static", 0.0, "retain_excluding_forgotten") \
    - at("static", top, "retain_excluding_forgotten") > 5
# Per seed: each seed trains its own gate, so the seeds differ from each other.
# What the paper's formula implies is that lambda changes nothing WITHIN a seed.
ROUTER_IGNORES_LAMBDA = bool(final[final["method"] == "router"]
                            .groupby("seed")["forget_all"].nunique().eq(1).all())
INTRAINING_KEEPS_MORE = at("intraining", 1.0, "retain_excluding_forgotten") \
    > at("static", 1.0, "retain_excluding_forgotten")
FORGETS_LESS_DEEPLY = at("intraining", top, "forget_current") > at("static", top, "forget_current")

for name, value in [("static fusion degrades as lambda grows", STATIC_DEGRADES),
                    ("router ignores lambda entirely", ROUTER_IGNORES_LAMBDA),
                    ("in-training keeps more than static at lambda 1.0", INTRAINING_KEEPS_MORE),
                    ("in-training forgets less deeply than static", FORGETS_LESS_DEEPLY)]:
    print(f"{'PASS' if value else 'FAIL'}  {name}")

### Plain-language summary

Every sentence below is assembled from the measured numbers above. Nothing is pre-written, so if
the run comes out differently the text changes with it.

In [ ]:
reserved = sorted(subspace["reserved_rank"].unique())
budget = D_IN // RANK
print(f"We deleted {len(CLASSES)} CIFAR-100 classes one at a time, {len(SEEDS)} times over.")
print()
print(f"Each deletion's patch used up {reserved if len(reserved) > 1 else reserved[0]} of the "
      f"{D_IN} directions the model reads from.")
print(f"At that rate the model runs out after {budget} deletions.")
print(f"The paper reports results at 20 and 30 deletions, and reports about 23 directions still")
print(f"free near deletion 16. Neither is reachable at this rate.")
print()
print(f"The paper's own method kept {at('intraining', 1.0, 'retain_excluding_forgotten'):.2f}% "
      f"of the classes we are keeping at lambda 1.0,")
print(f"against {at('static', 1.0, 'retain_excluding_forgotten'):.2f}% for plain subtraction, "
      f"from a starting point of {at('static', 0.0, 'retain_excluding_forgotten'):.2f}%.")
print("That is the paper's central claim and it holds here.")
print()
print(f"Projecting after training instead of during it left "
      f"{at('posthoc', 0.5, 'forget_current'):.2f}% accuracy on the class we asked it to delete,")
print(f"where training under the constraint left {at('intraining', 0.5, 'forget_current'):.2f}%. "
      f"When the projection happens decides whether it works.")
print()
print(f"The router scored {at('router', 1.0, 'forget_all'):.2f}% on the deleted classes, against "
      f"{at('static', 0.0, 'forget_all'):.2f}% for doing nothing at all.")
print("Read literally, its formula adds back what it was asked to remove.")

## 8. Save and verify

Writes the results, reads them back to confirm they survived the round trip, and uploads. A run
label is written once: uploading over an existing one stops rather than replacing the record an
earlier result was read from.

In [ ]:
metrics.to_csv(OUT / "metrics.csv", index=False)
subspace.to_csv(OUT / "subspace.csv", index=False)
norms.to_csv(OUT / "fusion_norms.csv", index=False)
table.to_csv(OUT / "summary_after_five_deletions.csv")

save_json(OUT / "metadata.json", {
    "notebook": "notebooks/03_cifar100_n5.ipynb",
    "run_label": RUN, "milestone": "M3",
    "spec_tests": ["T1", "T3"], "spec_discrepancies": ["D2", "D3", "D6", "D14", "D16"],
    "protocol": "CIFAR-100 N=5 sequential class deletion, four methods, lambda sweep, 3 seeds.",
    "interpretation_rule": (
        "T1: reserved rank 4 means the budget is 16 and Fig. 3 is unreachable; "
        "near 2.5 means Fig. 3 is right. Milestone rule: static degrades, router ignores "
        "lambda, in-training keeps more than static, in-training forgets less deeply."
    ),
    "limitations": REPORT["limitations"],
    "results": {
        "clean_accuracy_percent": REPORT["clean_accuracy_percent"],
        "reserved_ranks_observed": [int(v) for v in reserved],
        "measured_capacity_tasks": budget,
        "full_rank_every_task": bool(FULL_RANK),
        "static_degrades": bool(STATIC_DEGRADES),
        "router_ignores_lambda": bool(ROUTER_IGNORES_LAMBDA),
        "intraining_keeps_more": bool(INTRAINING_KEEPS_MORE),
        "intraining_forgets_less_deeply": bool(FORGETS_LESS_DEEPLY),
    },
    "config": {"cifar100": CFG["cifar100"], "unlearn": CFG["unlearn"],
               "projection": CFG["projection"], "lora": CFG["lora"], "model": CFG["model"]},
    "seeds": SEEDS, "provenance": PROV,
})

for path in sorted(OUT.iterdir()):
    print(f"{path.name:36s} {path.stat().st_size:>9,d} bytes")

assert len(pd.read_csv(OUT / "metrics.csv")) == len(metrics)
assert json.loads((OUT / "metadata.json").read_text(encoding="utf-8"))["run_label"] == RUN
print()
print("reopened and verified")

if STORE is None:
    print()
    print("Not uploaded: no R2 credentials, so this runtime holds the only copy.")
    print("These numbers cannot be cited in a trajectory entry until they are kept.")
else:
    for key in STORE.put(OUT, RUN):
        print("uploaded", key)

## 9. Release the GPU

Not applicable. This notebook never requested a GPU.